# Kimi Delta Attention (KDA) - a toy-scale build of Kimi K3

A minimal, from-scratch implementation of the **Kimi K3** architecture, from
*"Kimi K3: Open Frontier Intelligence"* (Kimi Team, 2026), built small enough
to train end-to-end on a free Colab GPU (or CPU) in a couple of minutes.

**How to read this notebook:** every section starts with a short markdown
explanation in plain words, followed by the code for that piece. Code
comments reference the paper's section and equation numbers directly, so you
can keep the report open next to this notebook and match them up line by
line.

**What this notebook is *not*:** a fast or production-scale implementation.
Every deliberate simplification is called out where it happens, along with
why it's a safe thing to simplify at this scale. See the repo's `README.md`
for the full list up front.

Full companion write-up: see `README.md` in this same folder - read that
first if you want the concepts before the code.

## 0. Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


## 1. The shape of the model, before any code

A K3 model is a stack of **blocks**. Every block repeats the same fixed
pattern of four sub-layers:

```
KDA  →  KDA  →  KDA  →  Gated MLA
```

So three linear-attention layers (KDA) for every one regular softmax
attention layer (Gated MLA). Every one of those four sub-layers is followed
by its own feed-forward network, and that feed-forward network is a
**Mixture-of-Experts**, not a plain MLP.

On top of that, the usual residual connection (`x = x + sublayer(x)`) is
replaced with **Attention Residuals**: a small learned attention mechanism
that mixes in *every* previous layer's output, not just the one right
before it.

Four pieces, four sections below:

| piece | stands in for | job |
|---|---|---|
| **KDA** | self-attention | linear attention with a per-channel forget gate — constant-size state instead of a growing KV cache |
| **Gated MLA** | — | ordinary softmax attention, kept once every 4 layers |
| **Stable LatentMoE** | a normal MLP | sparse feed-forward — only a few experts fire per token |
| **Attention Residuals** | `x + sublayer(x)` | lets each layer mix in *any* earlier layer's output, not just the last one |

We'll build small, reusable pieces first (normalization, a causal
convolution, two flavors of gated MLP), then the four main components, then
assemble them into the full model, then train it on a toy dataset to prove
it all actually works.

### 1.1 Config

Everything here is toy-scale on purpose: a 64-dimensional model, 2 attention
heads, a single block (so 3 KDA layers + 1 Gated MLA layer total), and a
handful of MoE experts. This is enough to exercise every mechanism in the
architecture without needing a real GPU cluster.

In [2]:
class K3Config:
    d_model = 64
    n_heads = 2
    d_head = 32                 # n_heads * d_head == d_model
    n_blocks = 1                 # each block = 3x KDA + 1x Gated MLA (s2.1)
    n_shared_experts = 2          # s2.3, Ns=2 in the paper
    n_routed_experts = 4
    top_k_experts = 2
    d_expert_latent = 32          # routed-expert latent width l (s2.3)
    gmin = -5.0                   # lower-bounded decay floor (s2.1.1, Eq.5)
    conv_kernel = 4
    max_seq_len = 32
    vocab_size = None             # set from the toy dataset later

## 2. Small building blocks used everywhere

Before the main components, four small reusable pieces:

- **RMSNorm** - a lighter-weight alternative to LayerNorm, used throughout
  K3 for normalizing activations.
- **ShortConv** - a small causal (left-padded) 1-D convolution, run
  separately over queries, keys, and values right before KDA's recurrence,
  so nearby tokens can mix locally before the long-range recurrence takes
  over.
- **SwiGLUExpert** - a standard gated MLP, used for the MoE's *shared*
  experts (the ones every token passes through).
- **SiTUExpert** - a smoothly-clipped gated MLP, used for the MoE's
  *routed* experts, chosen for extra numerical stability when many experts
  are stacked together (§2.3.2, Eq.12).

In [3]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

In [4]:
class ShortConv(nn.Module):
    '''Causal depthwise short convolution used ahead of q/k/v in KDA (s2.1.1, Eq.2).'''
    def __init__(self, dim, kernel_size=4):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(dim, dim, kernel_size, groups=dim, padding=0)
    def forward(self, x):  # x: [B,T,D]
        B, T, D = x.shape
        x = x.transpose(1, 2)                       # B,D,T
        x = F.pad(x, (self.kernel_size - 1, 0))      # left-pad only -> causal
        x = self.conv(x)
        return x.transpose(1, 2)                     # B,T,D

In [5]:
class SwiGLUExpert(nn.Module):
    '''Full-width gated FFN used for MoE shared experts.'''
    def __init__(self, d_in, d_out, hidden_mult=2):
        super().__init__()
        h = d_in * hidden_mult
        self.Wg = nn.Linear(d_in, h, bias=False)
        self.Wu = nn.Linear(d_in, h, bias=False)
        self.Wd = nn.Linear(h, d_out, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

In [6]:
class SiTUExpert(nn.Module):
    '''SiTU-GLU FFN (2.3.2, Eq.12) used for MoE routed experts (compact latent width).'''
    def __init__(self, d, hidden_mult=2, beta1=4.0, beta2=25.0):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
        self.beta1, self.beta2 = beta1, beta2
    def forward(self, x):
        g, u = self.Wg(x), self.Wu(x)
        gate = self.beta1 * torch.tanh(g / self.beta1) * torch.sigmoid(g)
        up   = self.beta2 * torch.tanh(u / self.beta2)
        return self.Wd(gate * up)

## 3. Kimi Delta Attention (KDA) - the core idea

**Why not just use normal attention everywhere?** Normal (softmax) attention
has to look back at *every* previous token to produce the next one, so its
memory footprint (the "KV cache") grows with sequence length. **Linear
attention** fixes this by keeping a fixed-size running summary, a small
matrix called the **state**, and updating it token by token instead of
storing every token individually.

**The problem with plain linear attention:** the state just keeps *adding*
new information forever, so it never forgets and gets muddier the longer the
sequence runs.

**KDA's fix - a forget gate on the state.** At every timestep, before
writing new information in, the model first *decays* what's already stored.
Same idea as an LSTM's forget gate, just applied to a matrix-shaped state.

### What happens at every single timestep

Picture the state `S` as a scratchpad matrix carried forward one token at a
time. At each new token `t`:

1. **Decay the scratchpad** - multiply the whole state by a decay value
   `alpha_t`, one value *per channel* (not one single number for the whole
   matrix - this is the "channel-wise" part of KDA). Close to 1 means "keep
   almost everything," close to 0 means "forget almost everything."
2. **Erase the part of the scratchpad tied to the current key**, scaled by a
   learned strength `beta_t`. This is the "delta rule": instead of blindly
   piling new information on top of old, the model first removes whatever it
   previously wrote for a similar key.
3. **Write the new value in**, again scaled by `beta_t`.
4. **Read out** an output by querying the freshly updated scratchpad with
   the current query.

Steps 1–4 below are exactly the four lines inside the `for t in range(T):`
loop - each tagged with its equation number.

The per-channel decay `alpha_t` isn't fixed - it's *predicted from the
input* through a small bottleneck (`alpha_down` → `alpha_up`), squashed with
a sigmoid, and floored at a minimum (`gmin`) so it can never decay to
*exactly* zero and wipe the state instantly. This is what makes KDA
"selective": it decides, per token and per channel, how much of the past
still matters.

> **Simplification used here:** the real KDA layer is implemented with a
> *chunkwise-parallel* algorithm - processing the sequence in chunks with
> batched matrix multiplications, which is what makes it fast on a GPU. This
> notebook instead uses the plain **sequential recurrence** (a Python `for`
> loop over every timestep) from Eq. 1–6 directly. Same math, same result,
> much slower, much easier to read - you can watch the state update one
> token at a time instead of reasoning about a batched chunk algorithm.

In [7]:
class KDA(nn.Module):
    def __init__(self, d_model, n_heads, d_head, gmin=-5.0, conv_kernel=4):
        super().__init__()
        self.n_heads, self.d_head, self.gmin = n_heads, d_head, gmin
        inner = n_heads * d_head

        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.q_conv = ShortConv(inner, conv_kernel)
        self.k_conv = ShortConv(inner, conv_kernel)
        self.v_conv = ShortConv(inner, conv_kernel)

        self.beta_proj = nn.Linear(d_model, n_heads, bias=True)   # scalar-per-head beta_t

        r = max(8, inner // 4)                                     # low-rank decay logits (Eq.2)
        self.alpha_down = nn.Linear(d_model, r, bias=False)
        self.alpha_up = nn.Linear(r, inner, bias=False)
        self.alpha_bias = nn.Parameter(torch.zeros(inner))
        self.A_log_scale = nn.Parameter(torch.zeros(n_heads))      # per-head A_h, init 0 (Eq.5)

        self.out_norm = RMSNorm(d_head)                            # head-wise RMSNorm (Eq.6)
        self.gate_proj = nn.Linear(d_model, inner, bias=True)      # full-rank output gate
        self.out_proj = nn.Linear(inner, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.n_heads, self.d_head

        q = F.silu(self.q_conv(self.q_proj(x))).view(B, T, H, Dh)
        k = F.silu(self.k_conv(self.k_proj(x))).view(B, T, H, Dh)
        v = F.silu(self.v_conv(self.v_proj(x))).view(B, T, H, Dh)
        q = F.normalize(q, p=2, dim=-1)
        k = F.normalize(k, p=2, dim=-1)

        beta = torch.sigmoid(self.beta_proj(x))                    # B,T,H

        z = (self.alpha_up(self.alpha_down(x)) + self.alpha_bias).view(B, T, H, Dh)
        A = self.A_log_scale.view(1, 1, H, 1)
        g = self.gmin * torch.sigmoid(torch.exp(A) * z)            # Eq.5
        alpha = torch.exp(g)                                       # channel-wise decay, in (e^gmin, 1)

        S = x.new_zeros(B, H, Dh, Dh)                               # recurrent state, dk x dv
        outs = []
        for t in range(T):                                          # Eq.1, unrolled in time
            k_t, v_t, q_t = k[:, t], v[:, t], q[:, t]
            a_t, b_t = alpha[:, t], beta[:, t]

            S = a_t.unsqueeze(-1) * S                                # step 1: decay -> Diag(alpha_t) S_{t-1}
            kv_proj = torch.einsum('bhd,bhde->bhe', k_t, S)          # k_t^T S
            S = S - b_t.view(B, H, 1, 1) * k_t.unsqueeze(-1) * kv_proj.unsqueeze(-2)  # step 2: erase
            S = S + b_t.view(B, H, 1, 1) * k_t.unsqueeze(-1) * v_t.unsqueeze(-2)      # step 3: write

            o_t = torch.einsum('bhd,bhde->bhe', q_t, S)              # step 4: read -> S_t^T q_t
            outs.append(o_t)

        o = self.out_norm(torch.stack(outs, dim=1)).reshape(B, T, H * Dh)
        gate = torch.sigmoid(self.gate_proj(x))
        return self.out_proj(gate * o)                               # Eq.6

## 4. Gated MLA - the "normal attention" layer

Every 4th layer, K3 steps back and uses regular softmax attention instead of
linear attention, with two notable choices:

- **No positional encoding (NoPE).** No RoPE, no learned position
  embeddings - the model leans on the KDA layers and the causal mask for
  positional information.
- **Output gate.** Same pattern as KDA: the attention output is multiplied
  by a learned sigmoid gate before being projected back down (Eq.7).

> **Simplification used here:** the real layer also compresses keys and
> values into a small latent vector before attention (the "Latent" in
> Multi-head **L**atent **A**ttention), purely to shrink the KV cache in
> production serving. That's a memory optimization, not a change to what
> the layer computes, so it's dropped here - this is standard multi-head
> softmax attention plus the NoPE + output-gate choices above.

In [8]:
class GatedMLA(nn.Module):
    def __init__(self, d_model, n_heads, d_head):
        super().__init__()
        self.n_heads, self.d_head = n_heads, d_head
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.gate_proj = nn.Linear(d_model, inner, bias=True)
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.scale = d_head ** -0.5

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.n_heads, self.d_head
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, Dh).transpose(1, 2)

        attn = torch.einsum('bhtd,bhsd->bhts', q, k) * self.scale     # no positional encoding
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        attn = attn.masked_fill(mask, float('-inf')).softmax(dim=-1)
        o = torch.einsum('bhts,bhsd->bhtd', attn, v).transpose(1, 2).reshape(B, T, H * Dh)

        gate = torch.sigmoid(self.gate_proj(x))                        # Eq.7
        return self.out_proj(gate * o)

## 5. Stable LatentMoE — the feed-forward layer

Instead of one big MLP after every attention layer, K3 uses a
**Mixture-of-Experts** feed-forward network with two kinds of experts
running in parallel:

- **Shared experts** - a small, fixed number of experts that process
  *every* token, no routing. A stable, always-on baseline.
- **Routed experts** - a larger pool where each token is sent only to its
  **top-k** highest-scoring experts, chosen by a small router network.
  Different tokens can light up different experts.

The two outputs are simply added: `output = shared_output + routed_output`.

Two extra "stability" choices on the routed side (both from §2.3):

- Routed experts run in a **compressed latent space** - project down, run
  the expert, project back up, which keeps their cost small even with many
  experts.
- A **SiTU-GLU** activation (built above as `SiTUExpert`) plus an RMSNorm
  right before the final up-projection, both there to keep training
  numerically stable with many experts stacked.

> **Simplifications used here:**
> - **Routing.** The real system uses "Quantile Balancing" (§2.3.3) to keep
>   expert load balanced across a batch. This notebook uses plain top-k
>   routing with no load-balancing trick.
> - **Sparse dispatch.** In a real system, only the tokens routed to an
>   expert are actually sent to it. Here, *every* expert processes *every*
>   token, and the ones that weren't selected are simply multiplied by
>   zero afterward - same result, far less code, more FLOPs than
>   necessary, which is a completely fair trade at toy scale.

In [9]:
class StableLatentMoE(nn.Module):
    def __init__(self, d_model, d_latent, n_shared, n_routed, top_k, hidden_mult=2):
        super().__init__()
        self.n_routed, self.top_k = n_routed, top_k
        self.shared_experts = nn.ModuleList(
            [SwiGLUExpert(d_model, d_model, hidden_mult) for _ in range(n_shared)])
        self.down_proj = nn.Linear(d_model, d_latent, bias=False)      # W_down
        self.router = nn.Linear(d_model, n_routed, bias=False)
        self.routed_experts = nn.ModuleList(
            [SiTUExpert(d_latent, hidden_mult) for _ in range(n_routed)])
        self.pre_up_norm = RMSNorm(d_latent)                            # 2.3.1
        self.up_proj = nn.Linear(d_latent, d_model, bias=False)         # W_up

    def forward(self, x):
        B, T, D = x.shape
        xf = x.reshape(-1, D)

        shared_out = sum(e(xf) for e in self.shared_experts)            # Eq.11, shared term

        scores = torch.sigmoid(self.router(xf))                         # Eq.13 router
        topk_val, topk_idx = torch.topk(scores, self.top_k, dim=-1)
        topk_weight = topk_val / topk_val.sum(-1, keepdim=True).clamp_min(1e-9)
        full_weight = torch.zeros_like(scores).scatter(-1, topk_idx, topk_weight)  # dense, mostly 0

        z = self.down_proj(xf)                                          # routed latent (Eq.11)
        u = z.new_zeros(z.shape)
        for e_idx in range(self.n_routed):                               # dense-compute simplification
            u = u + full_weight[:, e_idx:e_idx + 1] * self.routed_experts[e_idx](z)

        routed_out = self.up_proj(self.pre_up_norm(u))                   # Eq.11
        return (shared_out + routed_out).reshape(B, T, D)

## 6. Attention Residuals - rethinking the residual stream

Normally, a transformer layer does:

```
x = x + sublayer(x)
```

which only ever looks at *the immediately preceding layer's output*. K3
replaces this with a small attention mechanism: at layer `l`, it looks back
at *every* previous layer's output (plus the original embedding) and
combines them with **learned attention weights**, instead of just adding the
last one.

Concretely: each layer owns a learned "pseudo-query" vector. That
pseudo-query is compared against all previous layer outputs (normalized
first), turned into attention weights with a softmax, and used to compute a
weighted mixture of all previous outputs (Eq.8-9). That mixture - not just
the raw previous layer's output - is what gets fed into the current layer.

Intuitively: instead of forcing information through a single fixed chain of
layers, the model gets to choose, per layer, *which* earlier layers' outputs
are actually useful right now.

> **Simplification used here:** the paper describes both a **Full** form
> (attend over *all* previous layers) and a **Block-partitioned** form (only
> attend within nearby blocks, for efficiency at large depth). This notebook
> only has a handful of layers, so there's no need to partition — it uses
> the Full form directly. Exact same math, just skipping an efficiency trick
> that only matters once you have many, many layers.

In [10]:
class AttnRes(nn.Module):
    def __init__(self, d_model, n_layers):
        super().__init__()
        self.pseudo_queries = nn.ParameterList(
            [nn.Parameter(torch.randn(d_model) * 0.02) for _ in range(n_layers)])
        self.norm = RMSNorm(d_model)

    def compute_h(self, layer_idx, history):
        # history: list of [B,T,D] tensors -- embedding + every previous layer's output
        w = self.pseudo_queries[layer_idx]
        V = torch.stack(history, dim=2)                    # B,T,N,D
        K = self.norm(V)                                    # phi(q,k) kernel numerator (Eq.9)
        scores = torch.einsum('btnd,d->btn', K, w)
        attn = scores.softmax(dim=-1)
        return torch.einsum('btn,btnd->btd', attn, V)        # h_l, Eq.9

## 7. Putting the four pieces together

A `SubBlock` is one attention layer (KDA or Gated MLA) paired with its own
`StableLatentMoE`. The full model interleaves KDA and Gated MLA in the 3:1
pattern, and routes every layer's input through `AttnRes` instead of a plain
residual add.

Notice the forward pass keeps a running `history` list - the embedding, then
every sub-block's output, and passes the whole list into `AttnRes` at every
layer. That's the mechanism from Section 6 in action.

In [11]:
class SubBlock(nn.Module):
    def __init__(self, cfg, layer_type):
        super().__init__()
        self.attn = (KDA(cfg.d_model, cfg.n_heads, cfg.d_head, cfg.gmin, cfg.conv_kernel)
                     if layer_type == 'kda' else GatedMLA(cfg.d_model, cfg.n_heads, cfg.d_head))
        self.moe = StableLatentMoE(cfg.d_model, cfg.d_expert_latent,
                                    cfg.n_shared_experts, cfg.n_routed_experts, cfg.top_k_experts)

class KimiK3Mini(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        layer_types = (['kda', 'kda', 'kda', 'mla']) * cfg.n_blocks   # 2.1, 3:1 KDA:MLA ratio
        self.sub_blocks = nn.ModuleList([SubBlock(cfg, t) for t in layer_types])
        self.attnres = AttnRes(cfg.d_model, len(self.sub_blocks))
        self.final_norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        history = [x]                                       # v_0 = h_1 = embedding (s2.2)
        for l, block in enumerate(self.sub_blocks):
            h_l = self.attnres.compute_h(l, history)          # AttnRes replaces the plain residual
            y = h_l + block.attn(h_l)
            y = y + block.moe(y)
            history.append(y)
        return self.lm_head(self.final_norm(history[-1]))

## 8. Proving it actually works

Everything above is only worth something if the gradients actually flow
through all of it correctly — the KDA recurrence, the MoE routing (with its
`topk` + `scatter`, which can silently break gradients if done wrong), and
the Attention Residuals. So the rest of this notebook:

1. builds a **tiny synthetic dataset** (no real corpus needed - a repeating
   `"0123456789ABCDEF"` string is enough to check the model can learn *any*
   structure at all),
2. runs **one forward + backward pass** as a sanity check, confirming the
   output shape is right and no gradient comes out `NaN`,
3. **trains for a few hundred steps**, and
4. **generates** from the trained model - if training worked, the output
   should show visible periodicity, echoing the repeating pattern it was
   trained on.

This is deliberately not a "real" training run, no validation set, no real
language, nowhere near convergence to anything meaningful. It exists purely
to catch architecture bugs, which is the whole point of a toy-scale build.

In [12]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

cfg = K3Config()
cfg.vocab_size = len(chars)

model = KimiK3Mini(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

Model built. Trainable parameters: 406,764


In [13]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:cfg.max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:cfg.max_seq_len + 1].unsqueeze(0).to(device)
logits0 = model(xb0)
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {cfg.max_seq_len}, {cfg.vocab_size}])")
loss0 = F.cross_entropy(logits0.view(-1, cfg.vocab_size), yb0.view(-1))
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

Sanity check -- logits shape: (1, 32, 16) (expect [1, 32, 16])
Sanity check -- initial loss: 3.0303, NaN grads: 0


In [14]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, cfg.max_seq_len, batch_size, device)
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, cfg.vocab_size), yb.view(-1))
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

Training on synthetic periodic sequence (verifies grads flow end-to-end)...
  step    0 | loss 3.0248
  step   50 | loss 0.0046
  step  100 | loss 0.0021
  step  150 | loss 0.0013
  step  200 | loss 0.0009
  step  250 | loss 0.0006
  step  299 | loss 0.0005


In [15]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        logits = model(idx)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

Generated (should show visible periodicity if training worked):
0123456789ABCDEF0123456789ABCDEF0123456789ABCDEF01234567


## 9. Where to go from here

Some things worth trying if you want to push on this further:

- **Swap the sequential recurrence for the chunkwise-parallel form** in
  `KDA.forward` - same math, but processes the sequence in blocks with
  batched matmuls instead of a Python loop. This is the change that would
  actually make it fast.
- **Add Quantile Balancing** to `StableLatentMoE`'s router so expert load
  stays even across a batch, instead of the plain top-k used here.
- **Switch `AttnRes` to the Block-partitioned form** and try it with many
  more layers - the Full form used here gets expensive once `n_layers` is
  large, since every layer attends over the entire history.
- **Bring back MoonViT** and feed the model image patches alongside text
  tokens, to see the native vision pathway from the paper in action.

Reference:
1. Kimi Team, [*Kimi K3: Open Frontier Intelligence*](https://arxiv.org/pdf/2607.24653), 2026.
2. Kimi Team, [*Attention Residuals*](https://arxiv.org/pdf/2603.15031), 2026.
3. Kimi Team, [*Kimi Linear: An Expressive, Efficient Attention Architecture*](https://arxiv.org/pdf/2510.26692), 2025.